# Notebook 06 — Model Refinement, Calibration, and Clinical Utility Screening

**Project:** Machine Learning-Based Prediction of Parkinson’s Disease Progression Using PPMI Data

## Objective
Refine the baseline models from Notebook 05 using training-only hyperparameter tuning, evaluate tuned models on the held-out test set, assess calibration, generate threshold-level clinical utility summaries, and repeat the analysis without `baseline_NP3TOT` as a sensitivity analysis.

## Scientific Background
Notebook 05 showed that baseline models provide only modest discrimination. This notebook is therefore a refinement and validation step, not a final clinical model. The main goal is to determine whether tuned models improve performance while preserving strict separation between training-based model selection and held-out test evaluation.

## Dataset Verification
This notebook requires processed train/test matrices generated by Notebook 04:

- `06_train_processed_matrix.csv`
- `07_test_processed_matrix.csv`
- `08_processed_feature_names.csv`
- `11_processed_feature_names_without_baseline_NP3TOT.csv`

## Code
All preprocessing has already been completed in Notebook 04. This notebook does not refit imputation, scaling, or encoding pipelines.

## Scientific Interpretation
Model performance will be judged using ROC-AUC, PR-AUC, balanced accuracy, sensitivity, specificity, precision, F1, and Brier score. Accuracy alone is not sufficient because rapid progression is an imbalanced outcome.

## Quality Control Checklist
The notebook will confirm that:
- processed train/test files exist,
- both outcome classes are present,
- no missing values exist,
- tuning is restricted to training data,
- thresholds are selected from training predictions only,
- test data is used only once for final evaluation.

## Expected Output
The notebook saves tuned model summaries, test performance, calibration summaries, threshold tables, sensitivity analysis, feature importance, QC checklist, and a summary report.

In [ ]:
# ============================================================
# 01. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# 02. Imports and project paths
# ============================================================

from pathlib import Path
import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    ExtraTreesClassifier
)
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    roc_curve,
    precision_recall_curve
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.inspection import permutation_importance

import joblib

RANDOM_STATE = 42
TARGET_COL = "rapid_progression_q75"
ID_COL = "PATNO"

PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression")
NB04_DIR = PROJECT_DIR / "outputs" / "notebook_04_preprocessing"
OUT_DIR = PROJECT_DIR / "outputs" / "notebook_06_model_refinement_calibration"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("Notebook 04 output directory:", NB04_DIR)
print("Notebook 06 output directory:", OUT_DIR)
print("Notebook 04 directory exists:", NB04_DIR.exists())

if not NB04_DIR.exists():
    raise FileNotFoundError(
        f"Notebook 04 output directory not found: {NB04_DIR}\n"
        "Please run Notebook 04 first and confirm the path."
    )

In [ ]:
# ============================================================
# 03. Helper functions
# ============================================================

def safe_divide(a, b):
    return np.nan if b == 0 else a / b

def evaluate_binary_classifier(y_true, proba, threshold, model_name, evaluation_name):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba).astype(float)
    y_pred = (proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "model": model_name,
        "evaluation": evaluation_name,
        "threshold": float(threshold),
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc_average_precision": average_precision_score(y_true, proba),
        "brier_score": brier_score_loss(y_true, proba),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "sensitivity_recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": safe_divide(tn, tn + fp),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "positive_predictions": int(y_pred.sum()),
        "negative_predictions": int((1 - y_pred).sum()),
    }

def select_youden_threshold(y_true, proba):
    fpr, tpr, thresholds = roc_curve(y_true, proba)
    youden = tpr - fpr
    idx = int(np.argmax(youden))
    return float(thresholds[idx])

def threshold_table(y_true, proba, thresholds):
    rows = []
    for thr in thresholds:
        y_pred = (np.asarray(proba) >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        rows.append({
            "threshold": float(thr),
            "sensitivity_recall": safe_divide(tp, tp + fn),
            "specificity": safe_divide(tn, tn + fp),
            "precision_ppv": safe_divide(tp, tp + fp),
            "npv": safe_divide(tn, tn + fn),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "positive_predictions": int(y_pred.sum()),
            "negative_predictions": int((1 - y_pred).sum()),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        })
    return pd.DataFrame(rows)

def make_calibrated_classifier(estimator, method, cv):
    # Compatible with multiple scikit-learn versions.
    try:
        return CalibratedClassifierCV(estimator=estimator, method=method, cv=cv)
    except TypeError:
        return CalibratedClassifierCV(base_estimator=estimator, method=method, cv=cv)

def get_positive_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    scores = model.decision_function(X)
    return 1 / (1 + np.exp(-scores))

In [ ]:
# ============================================================
# 04. Load processed train/test matrices from Notebook 04
# ============================================================

required_files = {
    "train_processed": NB04_DIR / "06_train_processed_matrix.csv",
    "test_processed": NB04_DIR / "07_test_processed_matrix.csv",
    "processed_features": NB04_DIR / "08_processed_feature_names.csv",
    "sensitivity_features": NB04_DIR / "11_processed_feature_names_without_baseline_NP3TOT.csv",
}

for label, path in required_files.items():
    print(label, "->", path, "| exists:", path.exists())

missing = [str(p) for p in required_files.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required input files:\n" + "\n".join(missing))

train_df = pd.read_csv(required_files["train_processed"])
test_df = pd.read_csv(required_files["test_processed"])
feature_names = pd.read_csv(required_files["processed_features"])["processed_feature_name"].tolist()
sensitivity_feature_names = pd.read_csv(required_files["sensitivity_features"])["processed_feature_name"].tolist()

X_train = train_df[feature_names].copy()
y_train = train_df[TARGET_COL].astype(int).copy()
X_test = test_df[feature_names].copy()
y_test = test_df[TARGET_COL].astype(int).copy()

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train target distribution:")
display(y_train.value_counts().rename_axis("class").reset_index(name="n"))
print("Test target distribution:")
display(y_test.value_counts().rename_axis("class").reset_index(name="n"))

verification = pd.DataFrame([
    {"item": "train_rows", "value": X_train.shape[0]},
    {"item": "test_rows", "value": X_test.shape[0]},
    {"item": "processed_features", "value": X_train.shape[1]},
    {"item": "train_positive_n", "value": int(y_train.sum())},
    {"item": "test_positive_n", "value": int(y_test.sum())},
    {"item": "train_missing_values", "value": int(X_train.isna().sum().sum())},
    {"item": "test_missing_values", "value": int(X_test.isna().sum().sum())},
])
verification.to_csv(OUT_DIR / "01_dataset_verification.csv", index=False)
display(verification)

In [ ]:
# ============================================================
# 05. Define tuned model search spaces
# ============================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

model_spaces = {
    "Logistic_Regression_L2_Tuned": {
        "estimator": LogisticRegression(
            penalty="l2",
            solver="liblinear",
            max_iter=5000,
            random_state=RANDOM_STATE
        ),
        "params": {
            "C": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
            "class_weight": [None, "balanced"],
        },
        "n_iter": 14,
    },
    "Logistic_Regression_ElasticNet_Tuned": {
        "estimator": LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            max_iter=10000,
            random_state=RANDOM_STATE
        ),
        "params": {
            "C": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0],
            "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
            "class_weight": [None, "balanced"],
        },
        "n_iter": 18,
    },
    "Random_Forest_Tuned": {
        "estimator": RandomForestClassifier(
            n_estimators=400,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "params": {
            "max_depth": [3, 5, 8, None],
            "min_samples_leaf": [5, 10, 20, 30],
            "max_features": ["sqrt", 0.5, None],
            "class_weight": [None, "balanced", "balanced_subsample"],
        },
        "n_iter": 18,
    },
    "Extra_Trees_Tuned": {
        "estimator": ExtraTreesClassifier(
            n_estimators=400,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "params": {
            "max_depth": [3, 5, 8, None],
            "min_samples_leaf": [5, 10, 20, 30],
            "max_features": ["sqrt", 0.5, None],
            "class_weight": [None, "balanced"],
        },
        "n_iter": 18,
    },
    "Gradient_Boosting_Tuned": {
        "estimator": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "params": {
            "n_estimators": [50, 100, 150],
            "learning_rate": [0.01, 0.03, 0.05, 0.1],
            "max_depth": [2, 3],
            "min_samples_leaf": [5, 10, 20],
        },
        "n_iter": 18,
    },
    "Hist_Gradient_Boosting_Tuned": {
        "estimator": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        "params": {
            "max_iter": [50, 100, 150],
            "learning_rate": [0.01, 0.03, 0.05, 0.1],
            "max_leaf_nodes": [7, 15, 31],
            "l2_regularization": [0.0, 0.1, 1.0, 10.0],
        },
        "n_iter": 18,
    },
}

print("Models to tune:", list(model_spaces.keys()))

In [ ]:
# ============================================================
# 06. Hyperparameter tuning using training data only
# ============================================================

search_rows = []
best_estimators = {}

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "balanced_accuracy": "balanced_accuracy",
}

for model_name, spec in model_spaces.items():
    print("\nTuning:", model_name)

    search = RandomizedSearchCV(
        estimator=spec["estimator"],
        param_distributions=spec["params"],
        n_iter=spec["n_iter"],
        scoring=scoring,
        refit="roc_auc",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
        return_train_score=False,
        error_score=np.nan
    )

    search.fit(X_train, y_train)
    best_estimators[model_name] = search.best_estimator_

    row = {
        "model": model_name,
        "best_cv_roc_auc": search.best_score_,
        "best_params_json": json.dumps(search.best_params_),
    }

    # Add additional mean CV scores for the selected row.
    best_idx = search.best_index_
    for metric in ["average_precision", "balanced_accuracy"]:
        key = f"mean_test_{metric}"
        row[f"best_cv_{metric}"] = search.cv_results_[key][best_idx]

    search_rows.append(row)
    print("Best ROC-AUC:", round(search.best_score_, 4))
    print("Best params:", search.best_params_)

search_summary = pd.DataFrame(search_rows).sort_values(
    ["best_cv_roc_auc", "best_cv_average_precision", "best_cv_balanced_accuracy"],
    ascending=False
).reset_index(drop=True)

search_summary.insert(0, "rank", np.arange(1, len(search_summary) + 1))
search_summary.to_csv(OUT_DIR / "02_hyperparameter_search_summary.csv", index=False)
display(search_summary)

In [ ]:
# ============================================================
# 07. Training OOF performance for tuned models
# ============================================================

cv_perf_rows = []
thresholds = {}

for model_name, estimator in best_estimators.items():
    print("OOF prediction:", model_name)

    oof_proba = cross_val_predict(
        estimator,
        X_train,
        y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    thr = select_youden_threshold(y_train, oof_proba)
    thresholds[model_name] = thr

    cv_perf_rows.append(
        evaluate_binary_classifier(
            y_train, oof_proba, 0.5, model_name, "cv_oof_threshold_0.5"
        )
    )
    cv_perf_rows.append(
        evaluate_binary_classifier(
            y_train, oof_proba, thr, model_name, "cv_oof_training_derived_youden_threshold"
        )
    )

cv_perf = pd.DataFrame(cv_perf_rows)
cv_perf.to_csv(OUT_DIR / "03_tuned_cv_performance.csv", index=False)
display(cv_perf.sort_values(["evaluation", "roc_auc"], ascending=[True, False]))

In [ ]:
# ============================================================
# 08. Fit tuned models on full training set and evaluate held-out test set
# ============================================================

test_perf_rows = []
test_probas = {}

for model_name, estimator in best_estimators.items():
    model = clone(estimator)
    model.fit(X_train, y_train)

    proba = get_positive_proba(model, X_test)
    test_probas[model_name] = proba

    thr = thresholds[model_name]

    test_perf_rows.append(
        evaluate_binary_classifier(
            y_test, proba, 0.5, model_name, "test_threshold_0.5"
        )
    )
    test_perf_rows.append(
        evaluate_binary_classifier(
            y_test, proba, thr, model_name, "test_training_derived_youden_threshold"
        )
    )

    joblib.dump(model, OUT_DIR / f"model_{model_name}.joblib")

test_perf = pd.DataFrame(test_perf_rows)
test_perf.to_csv(OUT_DIR / "04_tuned_test_performance.csv", index=False)
display(test_perf.sort_values(["evaluation", "roc_auc"], ascending=[True, False]))

In [ ]:
# ============================================================
# 09. Select tuned model
# ============================================================

# Primary selection remains based on training OOF performance to avoid using the test set for model selection.
selection = cv_perf[
    cv_perf["evaluation"] == "cv_oof_training_derived_youden_threshold"
].copy()

selection = selection.sort_values(
    ["roc_auc", "pr_auc_average_precision", "balanced_accuracy"],
    ascending=False
).reset_index(drop=True)

selection.insert(0, "rank", np.arange(1, len(selection) + 1))
selection.to_csv(OUT_DIR / "05_tuned_model_ranking.csv", index=False)
display(selection)

selected_model_name = selection.loc[0, "model"]
selected_threshold = float(selection.loc[0, "threshold"])
selected_model = joblib.load(OUT_DIR / f"model_{selected_model_name}.joblib")
selected_test_proba = test_probas[selected_model_name]

with open(OUT_DIR / "06_selected_tuned_model.txt", "w") as f:
    f.write(f"Selected tuned model: {selected_model_name}\n")
    f.write(f"Training-derived threshold: {selected_threshold}\n")
    f.write("Selection rule: highest training OOF ROC-AUC, then PR-AUC, then balanced accuracy.\n")

print("Selected tuned model:", selected_model_name)
print("Selected threshold:", selected_threshold)

In [ ]:
# ============================================================
# 10. Calibration of selected tuned model using training data only
# ============================================================

calibration_rows = []

for method in ["sigmoid", "isotonic"]:
    print("Fitting calibrated model:", method)

    calibrated = make_calibrated_classifier(
        estimator=clone(best_estimators[selected_model_name]),
        method=method,
        cv=cv
    )
    calibrated.fit(X_train, y_train)

    cal_proba_test = calibrated.predict_proba(X_test)[:, 1]

    calibration_rows.append(
        evaluate_binary_classifier(
            y_test,
            cal_proba_test,
            0.5,
            f"{selected_model_name}_calibrated_{method}",
            "test_threshold_0.5_calibrated"
        )
    )

    joblib.dump(calibrated, OUT_DIR / f"model_{selected_model_name}_calibrated_{method}.joblib")

calibration_perf = pd.DataFrame(calibration_rows)
calibration_perf.to_csv(OUT_DIR / "07_calibrated_test_performance.csv", index=False)
display(calibration_perf)

In [ ]:
# ============================================================
# 11. Calibration curve for selected tuned model
# ============================================================

prob_true, prob_pred = calibration_curve(y_test, selected_test_proba, n_bins=5, strategy="quantile")

calibration_curve_df = pd.DataFrame({
    "mean_predicted_probability": prob_pred,
    "observed_fraction_positive": prob_true,
})
calibration_curve_df.to_csv(OUT_DIR / "08_calibration_curve_selected_model_test.csv", index=False)
display(calibration_curve_df)

plt.figure(figsize=(6, 5))
plt.plot(prob_pred, prob_true, marker="o", label=selected_model_name)
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed fraction positive")
plt.title("Calibration curve — selected tuned model")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_calibration_selected_tuned_model_test.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# 12. ROC and PR curves for tuned models on held-out test set
# ============================================================

plt.figure(figsize=(7, 6))
for model_name, proba in test_probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_value = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f"{model_name} AUC={auc_value:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves — tuned models on test set")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_roc_curves_tuned_models_test.png", dpi=300)
plt.show()

plt.figure(figsize=(7, 6))
baseline_prevalence = y_test.mean()
for model_name, proba in test_probas.items():
    precision, recall, _ = precision_recall_curve(y_test, proba)
    ap_value = average_precision_score(y_test, proba)
    plt.plot(recall, precision, label=f"{model_name} AP={ap_value:.3f}")
plt.axhline(baseline_prevalence, linestyle="--", label=f"Prevalence={baseline_prevalence:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curves — tuned models on test set")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_pr_curves_tuned_models_test.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# 13. Threshold-level clinical utility screening
# ============================================================

thresholds_to_check = sorted(set(
    [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, selected_threshold, 0.40, 0.50, 0.60, 0.70]
))

threshold_summary = threshold_table(y_test, selected_test_proba, thresholds_to_check)
threshold_summary.insert(0, "model", selected_model_name)
threshold_summary.to_csv(OUT_DIR / "09_decision_threshold_table_selected_model_test.csv", index=False)
display(threshold_summary)

In [ ]:
# ============================================================
# 14. Permutation importance on held-out test set
# ============================================================

# This is test-set interpretability for the selected model; it should be interpreted cautiously.
# It is not used for model fitting or threshold selection.

perm = permutation_importance(
    selected_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "feature": feature_names,
    "importance_mean": perm.importances_mean,
    "importance_sd": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

perm_df.to_csv(OUT_DIR / "10_permutation_importance_selected_model_test.csv", index=False)
display(perm_df.head(25))

In [ ]:
# ============================================================
# 15. Sensitivity analysis without baseline_NP3TOT
# ============================================================

sens_features = [c for c in sensitivity_feature_names if c in X_train.columns and c in X_test.columns]

X_train_sens = X_train[sens_features].copy()
X_test_sens = X_test[sens_features].copy()

print("Primary feature count:", X_train.shape[1])
print("Sensitivity feature count:", X_train_sens.shape[1])

sens_rows = []

# Reuse the same model spaces, but tune only on the sensitivity feature set.
sens_best_estimators = {}

for model_name, spec in model_spaces.items():
    print("\nSensitivity tuning:", model_name)

    search = RandomizedSearchCV(
        estimator=spec["estimator"],
        param_distributions=spec["params"],
        n_iter=spec["n_iter"],
        scoring=scoring,
        refit="roc_auc",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
        return_train_score=False,
        error_score=np.nan
    )

    search.fit(X_train_sens, y_train)
    sens_best_estimators[model_name] = search.best_estimator_

    oof_proba = cross_val_predict(
        search.best_estimator_,
        X_train_sens,
        y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    thr = select_youden_threshold(y_train, oof_proba)

    fitted = clone(search.best_estimator_)
    fitted.fit(X_train_sens, y_train)
    test_proba = fitted.predict_proba(X_test_sens)[:, 1]

    row = evaluate_binary_classifier(
        y_test,
        test_proba,
        thr,
        model_name,
        "sensitivity_test_no_baseline_NP3TOT_training_derived_youden_threshold"
    )
    row["best_params_json"] = json.dumps(search.best_params_)
    sens_rows.append(row)

sensitivity_results = pd.DataFrame(sens_rows).sort_values(
    ["roc_auc", "pr_auc_average_precision", "balanced_accuracy"],
    ascending=False
).reset_index(drop=True)

sensitivity_results.insert(0, "rank", np.arange(1, len(sensitivity_results) + 1))
sensitivity_results.to_csv(OUT_DIR / "11_sensitivity_no_baseline_NP3TOT_tuned_test_performance.csv", index=False)
display(sensitivity_results)

In [ ]:
# ============================================================
# 16. Quality Control Checklist
# ============================================================

qc_items = []

def add_qc(item, status, detail):
    qc_items.append({"qc_item": item, "status": status, "detail": detail})

add_qc(
    "Notebook 04 processed files found",
    "PASS" if all(p.exists() for p in required_files.values()) else "FAIL",
    "All required Notebook 04 processed files were located."
)

add_qc(
    "Feature/target row counts match",
    "PASS" if (len(X_train) == len(y_train) and len(X_test) == len(y_test)) else "FAIL",
    f"Train X/y: {len(X_train)}/{len(y_train)}; Test X/y: {len(X_test)}/{len(y_test)}"
)

add_qc(
    "Both classes present in train and test",
    "PASS" if (y_train.nunique() == 2 and y_test.nunique() == 2) else "FAIL",
    f"Train classes: {sorted(y_train.unique())}; Test classes: {sorted(y_test.unique())}"
)

add_qc(
    "No missing values in processed matrices",
    "PASS" if (X_train.isna().sum().sum() == 0 and X_test.isna().sum().sum() == 0) else "FAIL",
    f"Train missing={int(X_train.isna().sum().sum())}; Test missing={int(X_test.isna().sum().sum())}"
)

add_qc(
    "Hyperparameter tuning restricted to training set",
    "PASS",
    "RandomizedSearchCV was fitted using X_train and y_train only."
)

add_qc(
    "Thresholds selected without test leakage",
    "PASS",
    "Thresholds were derived from out-of-fold predictions within the training set."
)

add_qc(
    "Held-out test evaluation completed",
    "PASS" if test_perf.shape[0] > 0 else "FAIL",
    f"Tuned models evaluated: {len(best_estimators)}"
)

add_qc(
    "Calibration assessed",
    "PASS" if calibration_perf.shape[0] > 0 else "FAIL",
    "Sigmoid and isotonic calibration were fitted using training data only."
)

add_qc(
    "Sensitivity analysis excluding baseline_NP3TOT completed",
    "PASS" if sensitivity_results.shape[0] > 0 else "FAIL",
    f"Sensitivity feature count: {len(sens_features)}"
)

add_qc(
    "No preprocessing refit in Notebook 06",
    "PASS",
    "Notebook 06 uses processed matrices from Notebook 04 and does not refit imputation/scaling/encoding."
)

qc = pd.DataFrame(qc_items)
qc.to_csv(OUT_DIR / "12_quality_control_checklist.csv", index=False)
display(qc)

if (qc["status"] == "FAIL").any():
    raise RuntimeError("One or more QC checks failed. Review 12_quality_control_checklist.csv.")

In [ ]:
# ============================================================
# 17. Summary report
# ============================================================

selected_test_row = test_perf[
    (test_perf["model"] == selected_model_name)
    & (test_perf["evaluation"] == "test_training_derived_youden_threshold")
].iloc[0].to_dict()

best_sensitivity_row = sensitivity_results.iloc[0].to_dict()

summary = f'''
Notebook 06 — Model Refinement, Calibration, and Clinical Utility Screening
================================================================================

Input source:
{NB04_DIR}

Dataset:
- Train n: {len(y_train)}
- Test n: {len(y_test)}
- Processed predictors: {X_train.shape[1]}
- Train positive n: {int(y_train.sum())} ({100*y_train.mean():.2f}%)
- Test positive n: {int(y_test.sum())} ({100*y_test.mean():.2f}%)

Selected tuned model:
- {selected_model_name}
- Training-derived threshold: {selected_threshold:.6f}
- Selection rule: highest training OOF ROC-AUC, then PR-AUC, then balanced accuracy.

Held-out test performance for selected tuned model:
- roc_auc: {selected_test_row["roc_auc"]:.4f}
- pr_auc_average_precision: {selected_test_row["pr_auc_average_precision"]:.4f}
- balanced_accuracy: {selected_test_row["balanced_accuracy"]:.4f}
- sensitivity_recall: {selected_test_row["sensitivity_recall"]:.4f}
- specificity: {selected_test_row["specificity"]:.4f}
- precision: {selected_test_row["precision"]:.4f}
- f1: {selected_test_row["f1"]:.4f}
- brier_score: {selected_test_row["brier_score"]:.4f}

Best sensitivity model without baseline_NP3TOT:
- {best_sensitivity_row["model"]}
- roc_auc: {best_sensitivity_row["roc_auc"]:.4f}
- pr_auc_average_precision: {best_sensitivity_row["pr_auc_average_precision"]:.4f}
- balanced_accuracy: {best_sensitivity_row["balanced_accuracy"]:.4f}

Methodological notes:
- Hyperparameter tuning was performed within the training set only.
- Test-set outcomes were not used to tune models or select thresholds.
- Calibration was assessed, but this remains internal validation only.
- These models are not clinical tools without external validation.

Generated files:
- 01_dataset_verification.csv
- 02_hyperparameter_search_summary.csv
- 03_tuned_cv_performance.csv
- 04_tuned_test_performance.csv
- 05_tuned_model_ranking.csv
- 06_selected_tuned_model.txt
- 07_calibrated_test_performance.csv
- 08_calibration_curve_selected_model_test.csv
- 09_decision_threshold_table_selected_model_test.csv
- 10_permutation_importance_selected_model_test.csv
- 11_sensitivity_no_baseline_NP3TOT_tuned_test_performance.csv
- 12_quality_control_checklist.csv
- fig_calibration_selected_tuned_model_test.png
- fig_roc_curves_tuned_models_test.png
- fig_pr_curves_tuned_models_test.png

Output folder:
{OUT_DIR}
'''.strip()

print(summary)

with open(OUT_DIR / "13_notebook_06_summary_report.txt", "w") as f:
    f.write(summary)